# PBIT Final Evaluation — Test Set

Two candidate models trained on the **full training set** and evaluated **once** on the held-out test set.

| Model | Config | Rationale |
|-------|--------|-----------|
| LightGBM | Aug + Freq×Sev + f05_boost | Best CV model: Macro F1=0.427, F06=0.667, Sev3=0.833 |
| XGBoost  | Sev weights + f05_boost    | Best test result so far: F06=1.0, F05=0.45 |

## Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb

from pbit.modeling import (
    META_COLS, FAULT_ORDER,
    XGBWrapper,
    compute_sample_weights, evaluate_predictions,
)
from pbit.data_validation import load_long_dataset
from pbit.augmentation import augment_training_set
from pbit.features import extract_features

TRAIN_PATH   = PROJECT_ROOT / 'data' / 'train_real.csv'
TEST_PATH    = PROJECT_ROOT / 'data' / 'test_real.csv'
LONG_PATH    = PROJECT_ROOT / 'data' / 'PBIT_atuador_DESCARAC_r1_long_clean.csv'
LGBM_PARAMS  = PROJECT_ROOT / 'data' / 'best_lgbm_params.json'
XGB_PARAMS   = PROJECT_ROOT / 'data' / 'best_xgb_params.json'
RESULTS_PATH = PROJECT_ROOT / 'data' / 'final_eval_results.csv'

PROJECT_ROOT

## Load Data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
long_df  = load_long_dataset(LONG_PATH)

feat_cols = [c for c in train_df.columns if c not in META_COLS]

print('Training set :', len(train_df), 'experiments x', len(feat_cols), 'features')
print('Test set     :', len(test_df),  'experiments')
print('Long dataset :', long_df['Experiment_Index'].nunique(), 'experiments')
print()
print('Train class distribution:')
print(train_df['Fault_Mode'].value_counts().reindex(FAULT_ORDER).to_string())
print()
print('Test class distribution:')
print(test_df['Fault_Mode'].value_counts().reindex(FAULT_ORDER).to_string())

X_test = test_df[feat_cols].values.astype(float)
y_test = test_df['Fault_Mode'].values

---
## Model 1 — LightGBM Aug + Freq×Sev

Best model in CV: Macro F1=0.427, F06=0.667, Sev3=0.833 (5-fold grouped CV).
Augments F04/F05/F06 to 160 each with 1% Gaussian noise.
Applies frequency×severity sample weights + f05_boost=7x.

In [ ]:
with open(LGBM_PARAMS) as f:
    lgbm_all = json.load(f)

f05_boost_lgb = lgbm_all.pop('f05_boost')
lgbm_params   = lgbm_all  # remaining keys are LightGBM hyperparameters

# Augment full training set (F04/F05/F06 → target 160)
train_exp_ids = set(train_df['Experiment_Index'].tolist())
aug_long = augment_training_set(
    long_df, train_exp_ids,
    target_per_class=160, noise_scale=0.01, random_state=42,
)
aug_feat = extract_features(aug_long)
tr_full  = pd.concat([train_df, aug_feat], ignore_index=True)

X_train_aug = tr_full[feat_cols].values.astype(float)
y_train_aug = tr_full['Fault_Mode'].values

# Freq x Sev weights + F05 boost
sw_lgb = compute_sample_weights(y_train_aug, 'freq_sev')
sw_lgb[y_train_aug == 'F05'] *= f05_boost_lgb

lgb_model = lgb.LGBMClassifier(**lgbm_params)
lgb_model.fit(X_train_aug, y_train_aug, sample_weight=sw_lgb)
lgb_pred = lgb_model.predict(X_test)

lgb_metrics = evaluate_predictions(y_test, lgb_pred)
print('LightGBM Aug+Freq*Sev — Test Set')
print('Macro F1     :', round(lgb_metrics['macro_f1'], 4))
print('Balanced Acc :', round(lgb_metrics['balanced_accuracy'], 4))
print('F06 Recall   :', round(lgb_metrics['f06_recall'], 4))
print('F05 Recall   :', round(lgb_metrics['f05_recall'], 4))
print('F04 Recall   :', round(lgb_metrics['f04_recall'], 4))
print('Sev3 Recall  :', round(lgb_metrics['sev3_recall'], 4))

---
## Model 2 — XGBoost Sev weights

Best test-set performance observed so far: F06=1.0, F05=0.45, Balanced Acc=0.528.
Applies severity-based sample weights + f05_boost (no augmentation).

In [ ]:
with open(XGB_PARAMS) as f:
    xgb_all = json.load(f)

f05_boost_xgb = xgb_all.pop('f05_boost')
xgb_params    = xgb_all

# No augmentation — train on real data only
X_train = train_df[feat_cols].values.astype(float)
y_train = train_df['Fault_Mode'].values

sw_xgb = compute_sample_weights(y_train, 'sev')
sw_xgb[y_train == 'F05'] *= f05_boost_xgb

xgb_model = XGBWrapper(**xgb_params)
xgb_model.fit(X_train, y_train, sample_weight=sw_xgb)
xgb_pred = xgb_model.predict(X_test)

xgb_metrics = evaluate_predictions(y_test, xgb_pred)
print('XGBoost Sev weights — Test Set')
print('Macro F1     :', round(xgb_metrics['macro_f1'], 4))
print('Balanced Acc :', round(xgb_metrics['balanced_accuracy'], 4))
print('F06 Recall   :', round(xgb_metrics['f06_recall'], 4))
print('F05 Recall   :', round(xgb_metrics['f05_recall'], 4))
print('F04 Recall   :', round(xgb_metrics['f04_recall'], 4))
print('Sev3 Recall  :', round(xgb_metrics['sev3_recall'], 4))

---

In [ ]:
models_eval = [
    ('LightGBM Aug+Freq*Sev', lgb_metrics),
    ('XGBoost Sev weights',   xgb_metrics),
]

rows = []
for name, m in models_eval:
    rows.append({
        'Model':      name,
        'Macro_F1':   round(m['macro_f1'], 4),
        'Bal_Acc':    round(m['balanced_accuracy'], 4),
        'F06_Recall': round(m['f06_recall'], 4),
        'F05_Recall': round(m['f05_recall'], 4),
        'F04_Recall': round(m['f04_recall'], 4),
        'Sev3_Recall':round(m['sev3_recall'], 4),
    })

results_df = pd.DataFrame(rows)
results_df

### LightGBM Aug+Freq*Sev — Confusion Matrix

In [ ]:
cm_labels = lgb_metrics['cm_labels']
cm_lgb_df = pd.DataFrame(lgb_metrics['confusion_matrix'], index=cm_labels, columns=cm_labels)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm_lgb_df, annot=True, fmt='d', cmap='Blues', linewidths=0.4, ax=ax)
ax.set_title('LightGBM Aug+Freq*Sev — Test Set Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
plt.show()

print('Per-class recall (LightGBM):')
for cls in FAULT_ORDER:
    r = lgb_metrics['per_class_recall'].get(cls, float('nan'))
    print('  ' + cls + ':', round(r, 3))

### XGBoost Sev weights — Confusion Matrix

In [ ]:
cm_labels_x = xgb_metrics['cm_labels']
cm_xgb_df = pd.DataFrame(xgb_metrics['confusion_matrix'], index=cm_labels_x, columns=cm_labels_x)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm_xgb_df, annot=True, fmt='d', cmap='Oranges', linewidths=0.4, ax=ax)
ax.set_title('XGBoost Sev weights — Test Set Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
plt.show()

print('Per-class recall (XGBoost):')
for cls in FAULT_ORDER:
    r = xgb_metrics['per_class_recall'].get(cls, float('nan'))
    print('  ' + cls + ':', round(r, 3))

### Per-class Recall

In [ ]:
classes = FAULT_ORDER
lgb_rec = [lgb_metrics['per_class_recall'].get(c, float('nan')) for c in classes]
xgb_rec = [xgb_metrics['per_class_recall'].get(c, float('nan')) for c in classes]

x = np.arange(len(classes))
w = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, lgb_rec, w, label='LightGBM Aug+Freq*Sev', color='steelblue',  alpha=0.85)
ax.bar(x + w/2, xgb_rec, w, label='XGBoost Sev weights',   color='darkorange', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Recall')
ax.set_title('Per-class Recall — Test Set')
ax.legend()
ax.grid(True, axis='y', linewidth=0.3)
for xi, (lb, xb) in enumerate(zip(lgb_rec, xgb_rec)):
    if lb == lb: ax.text(xi - w/2, lb + 0.02, str(round(lb, 2)), ha='center', fontsize=7)
    if xb == xb: ax.text(xi + w/2, xb + 0.02, str(round(xb, 2)), ha='center', fontsize=7)
plt.tight_layout()
plt.show()

### Severity-level Recall

In [ ]:
sev_map = [
    ('Sev0 (N01,N02)',      'sev0_recall'),
    ('Sev1 (F01)',           'sev1_recall'),
    ('Sev2 (F02,F04,F05)',  'sev2_recall'),
    ('Sev3 (F03,F06)',      'sev3_recall'),
]
sev_df = pd.DataFrame([{
    'Severity': label,
    'LightGBM': round(lgb_metrics.get(key, float('nan')), 4),
    'XGBoost':  round(xgb_metrics.get(key, float('nan')), 4),
} for label, key in sev_map])
sev_df

### Feature Importance — LightGBM

In [ ]:
feat_imp = pd.Series(
    lgb_model.feature_importances_,
    index=feat_cols,
    name='Importance',
)
top20 = feat_imp.nlargest(20)

fig, ax = plt.subplots(figsize=(9, 7))
top20[::-1].plot.barh(ax=ax, color='steelblue')
ax.set_title('LightGBM — Top 20 Feature Importances (split count)')
ax.set_xlabel('Importance')
ax.grid(True, axis='x', linewidth=0.3)
plt.tight_layout()
plt.show()

print('Top 20:')
print(top20.to_string())

### Error Analysis

In [ ]:
print('=== LightGBM Aug+Freq*Sev ===')
lgb_err = []
for cls in FAULT_ORDER:
    mask = y_test == cls
    n_total = int(mask.sum())
    if n_total == 0:
        continue
    n_correct = int(((lgb_pred == cls) & mask).sum())
    wrong = lgb_pred[mask & (lgb_pred != cls)]
    top_err = pd.Series(wrong).value_counts().head(2).to_dict() if len(wrong) else {}
    lgb_err.append({
        'Class': cls, 'Total': n_total, 'Correct': n_correct,
        'Errors': n_total - n_correct,
        'Recall': round(n_correct / n_total, 3),
        'Top_Predicted_As': str(top_err),
    })
lgb_err_df = pd.DataFrame(lgb_err)
print(lgb_err_df.to_string(index=False))

print()
print('=== XGBoost Sev weights ===')
xgb_err = []
for cls in FAULT_ORDER:
    mask = y_test == cls
    n_total = int(mask.sum())
    if n_total == 0:
        continue
    n_correct = int(((xgb_pred == cls) & mask).sum())
    wrong = xgb_pred[mask & (xgb_pred != cls)]
    top_err = pd.Series(wrong).value_counts().head(2).to_dict() if len(wrong) else {}
    xgb_err.append({
        'Class': cls, 'Total': n_total, 'Correct': n_correct,
        'Errors': n_total - n_correct,
        'Recall': round(n_correct / n_total, 3),
        'Top_Predicted_As': str(top_err),
    })
xgb_err_df = pd.DataFrame(xgb_err)
print(xgb_err_df.to_string(index=False))

---
## Safety Metrics

Three safety-oriented metrics derived from the confusion matrix:

| Metric | Definition |
|--------|-----------|
| **CMR** (Critical Miss Rate) | Rate at which severity-3 faults (F03, F06) are predicted as severity-0 (N01, N02) |
| **Downward Risk** | Rate at which any fault is predicted as a strictly lower-severity class |
| **Severity Error Rate** | Rate of any severity mismatch between true and predicted class |

In [ ]:
SEV = {'N01': 0, 'N02': 0, 'F01': 1, 'F02': 2, 'F03': 3, 'F04': 2, 'F05': 2, 'F06': 3}
SEV0 = {'N01', 'N02'}
SEV3 = {'F03', 'F06'}

def safety_metrics(y_true, y_pred):
    n = len(y_true)
    sev_true = np.array([SEV[c] for c in y_true])
    sev_pred = np.array([SEV[c] for c in y_pred])

    crit_mask = np.array([c in SEV3 for c in y_true])
    cmr_numer = np.sum(crit_mask & np.array([c in SEV0 for c in y_pred]))
    cmr = cmr_numer / crit_mask.sum() if crit_mask.sum() > 0 else float('nan')

    down_risk = np.sum(sev_pred < sev_true) / n
    sev_err   = np.sum(sev_pred != sev_true) / n

    return {
        'cmr':       round(cmr, 4),
        'down_risk': round(down_risk, 4),
        'sev_err':   round(sev_err, 4),
        'cmr_count': int(cmr_numer),
        'crit_total': int(crit_mask.sum()),
    }

sm_lgb = safety_metrics(y_test, lgb_pred)
sm_xgb = safety_metrics(y_test, xgb_pred)

print('Safety Metrics — Test Set')
print(f"{'Metric':<22} {'LightGBM':>12} {'XGBoost':>12}")
print('-' * 48)
print(f"{'CMR (lower=safer)':<22} {sm_lgb['cmr']:>12.4f} {sm_xgb['cmr']:>12.4f}")
print(f"{'Downward Risk':<22} {sm_lgb['down_risk']:>12.4f} {sm_xgb['down_risk']:>12.4f}")
print(f"{'Severity Error Rate':<22} {sm_lgb['sev_err']:>12.4f} {sm_xgb['sev_err']:>12.4f}")
print()
print(f"CMR detail — critical samples (F03+F06): {sm_lgb['crit_total']}")
print(f"  LightGBM predicted as Sev0: {sm_lgb['cmr_count']}")
print(f"  XGBoost  predicted as Sev0: {sm_xgb['cmr_count']}")

In [ ]:
crit_mask = np.array([c in SEV3 for c in y_test])
sev0_pred_lgb = np.array([c in SEV0 for c in lgb_pred])
sev0_pred_xgb = np.array([c in SEV0 for c in xgb_pred])

print('=== CMR Breakdown: Sev3 samples predicted as Sev0 ===')
print()
for model_name, sev0_mask, pred in [('LightGBM', sev0_pred_lgb, lgb_pred),
                                      ('XGBoost',  sev0_pred_xgb, xgb_pred)]:
    cmr_samples = np.where(crit_mask & sev0_mask)[0]
    print(f'{model_name} ({len(cmr_samples)} critical misses):')
    if len(cmr_samples) == 0:
        print('  None')
    else:
        breakdown = pd.DataFrame({
            'True':      y_test[cmr_samples],
            'Predicted': pred[cmr_samples],
        })
        print(breakdown.value_counts().reset_index().rename(columns={0: 'Count'}).to_string(index=False))
    print()

## Save Results

In [ ]:
rows_ext = []
for name, m, sm in [
    ('LightGBM Aug+Freq*Sev', lgb_metrics, sm_lgb),
    ('XGBoost Sev weights',   xgb_metrics, sm_xgb),
]:
    rows_ext.append({
        'Model':           name,
        'Macro_F1':        round(m['macro_f1'], 4),
        'Bal_Acc':         round(m['balanced_accuracy'], 4),
        'F06_Recall':      round(m['f06_recall'], 4),
        'F05_Recall':      round(m['f05_recall'], 4),
        'F04_Recall':      round(m['f04_recall'], 4),
        'Sev3_Recall':     round(m['sev3_recall'], 4),
        'CMR':             sm['cmr'],
        'Downward_Risk':   sm['down_risk'],
        'Severity_Err':    sm['sev_err'],
    })

results_df = pd.DataFrame(rows_ext)
results_df.to_csv(RESULTS_PATH, index=False)
print('Saved:', RESULTS_PATH)
results_df